# LoRA plumbing probe for Gemma 4 E2B

Trains a throwaway rank-16 LoRA adapter for about 20 steps on generic text and
uploads it as a private Kaggle dataset, so the ladder runner can check that the
official harness (vLLM 0.19.1, W4A16 compressed-tensors checkpoint) registers
an adapter and grades an episode with it. The adapter is not meant to help:
it only proves the plumbing.

- Base weights: `google/gemma-4/transformers/gemma-4-e2b-it-qat-q4_0-unquantized`,
  the unquantized quantization-aware-trained checkpoint (assumed source of the
  W4A16 weights; the Kaggle card does not name it).
- Training text: CPython standard-library source shipped with this runtime.
  No competition task, patch or test is used.
- Credentials: `KAGGLE_API_TOKEN` from Colab secrets. Never paste it here.
- Runtime: Colab Pro, L4 or A100.

In [ ]:
import os, sys, json, time, subprocess
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
# Colab ships torchao 0.10, which current peft rejects at import time; peft does not need it here
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle", "peft", "transformers", "accelerate", "safetensors"], check=True)
import torch, transformers, peft
print("torch", torch.__version__, "transformers", transformers.__version__, "peft", peft.__version__)
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

In [ ]:
OWNER = "d410qc"
BASE = "/content/base"
DATASET = f"{OWNER}/g4-lora-probe"
OUT = "/content/lora_probe"
if not os.path.exists(f"{BASE}/config.json"):
    subprocess.run(["kaggle", "models", "instances", "versions", "download",
                    "google/gemma-4/transformers/gemma-4-e2b-it-qat-q4_0-unquantized/2",
                    "-p", BASE, "--untar"], check=True)
print(sorted(os.listdir(BASE)))
print(json.load(open(f"{BASE}/config.json")).get("architectures"))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(BASE)
try:
    model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="cuda")
except Exception as exc:  # multimodal checkpoints may only load through the image-text class
    print("causal-LM class failed:", type(exc).__name__, exc)
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(BASE, dtype=torch.bfloat16, device_map="cuda")
print(type(model).__name__)
proj = [n for n, _ in model.named_modules() if n.endswith(("q_proj", "down_proj"))]
print(len(proj), proj[:4], proj[-2:])

In [ ]:
from peft import LoraConfig, get_peft_model
# language-model projections only: the vision and audio towers are left alone
TARGET = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
cfg = LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, target_modules=TARGET, task_type="CAUSAL_LM")
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

In [ ]:
import inspect, json as _json, argparse, textwrap
text = "\n\n".join(inspect.getsource(m) for m in (_json.decoder, argparse, textwrap))
ids = tok(text, return_tensors="pt").input_ids[0]
SEQ, STEPS = 512, 20
chunks = [ids[i:i + SEQ] for i in range(0, len(ids) - SEQ, SEQ)][:STEPS]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
model.train()
losses = []
t0 = time.time()
for step, chunk in enumerate(chunks):
    batch = chunk.unsqueeze(0).cuda()
    loss = model(input_ids=batch, labels=batch).loss
    loss.backward(); opt.step(); opt.zero_grad()
    losses.append(round(loss.item(), 4))
    print(step, losses[-1])
print("steps", len(losses), "seconds", round(time.time() - t0))

In [ ]:
model.save_pretrained(OUT, safe_serialization=True)
os.listdir(OUT)
from safetensors import safe_open
with safe_open(f"{OUT}/adapter_model.safetensors", "pt") as f:
    keys = list(f.keys())
print(len(keys), keys[:3])
acfg = json.load(open(f"{OUT}/adapter_config.json"))
print({k: acfg[k] for k in ("r", "lora_alpha", "target_modules", "base_model_name_or_path")})

In [ ]:
# upload: adapter files plus a training record, as a private dataset
record = {"base": "google/gemma-4/transformers/gemma-4-e2b-it-qat-q4_0-unquantized/2", "rank": 16, "alpha": 16,
          "target_modules": TARGET, "steps": len(losses), "seq_len": SEQ, "lr": 1e-4, "losses": losses,
          "text": "CPython stdlib source: json.decoder, argparse, textwrap", "n_tensors": len(keys),
          "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
          "gpu": torch.cuda.get_device_name(0)}
json.dump(record, open(f"{OUT}/train_record.json", "w"), indent=1)
json.dump({"title": "g4-lora-probe", "id": DATASET, "licenses": [{"name": "apache-2.0"}]},
          open(f"{OUT}/dataset-metadata.json", "w"))
exists = subprocess.run(["kaggle", "datasets", "status", DATASET], capture_output=True, text=True).returncode == 0
cmd = (["kaggle", "datasets", "version", "-p", OUT, "-m", "probe adapter"] if exists
       else ["kaggle", "datasets", "create", "-p", OUT])
print(subprocess.run(cmd, capture_output=True, text=True).stdout[-800:])
print(json.dumps(record, indent=1))